# 07 -- fact_fantasy_teams Seed (Schema Seed)

**Purpose**: Create the empty `fact_fantasy_teams` parquet with correct column types.
This is a schema seed only -- the draft notebook populates rows once picks are made.

**Key**: `team_key + player_key`

**Outputs**:
- `data/fact_fantasy_teams.parquet` (empty, typed schema)

In [ ]:
import pandas as pd
import numpy as np
import nflreadpy as nfl
from pathlib import Path
from datetime import datetime, date
from dataclasses import dataclass, field


@dataclass
class LeagueConfig:
    draft_year: int = 2026
    total_cap: int = 500_000_000
    num_teams: int = 28
    num_conferences: int = 2
    initial_contract_years: int = 3
    extension_contract_years: int = 3
    fa_minimum_salary: int = 2_000_000
    data_dir: str = "data"
    fuzzy_auto_threshold: int = 90
    fuzzy_review_threshold: int = 70


CFG = LeagueConfig()
DATA = Path(CFG.data_dir)
TODAY = date.today().isoformat()
DATA.mkdir(exist_ok=True)

## 1 -- Schema Definition

Nullable integer columns use pandas `Int64` (capital I) to support NA.

**Player FK columns**:
- `gsis_id` → `dim_nfl_players.gsis_id` (primary, post-signing)
- `player_key` → `dim_rookie_prospect.player_key` (interim, pre-signing window)

Other FKs: `team_key` → `dim_team`, `contract_id` → `dim_contract`.

In [ ]:
# -- fact_fantasy_teams schema seed -------------------------------------------------------
# gsis_id  : primary player FK -> dim_nfl_players.gsis_id
# player_key: interim FK -> dim_rookie_prospect.player_key (pre-signing window only)
fact_fantasy_teams = pd.DataFrame({
    "team_key":        pd.Series(dtype="str"),
    "gsis_id":         pd.Series(dtype="str"),    # FK -> dim_nfl_players (primary)
    "player_key":      pd.Series(dtype="str"),    # Interim FK -> dim_rookie_prospect
    "conference":      pd.Series(dtype="str"),    # A or B (dual-conference support)
    "contract_id":     pd.Series(dtype="str"),    # FK -> dim_contract.contract_id
    "contract_value":  pd.Series(dtype="Int64"),
    "contract_year":   pd.Series(dtype="Int64"),  # 1/2/3 within contract term
    "cap_hit":         pd.Series(dtype="Int64"),  # contract_value * cap_hit_pct
    "dead_money":      pd.Series(dtype="Int64"),  # cap hit remaining if dropped
    "status":          pd.Series(dtype="str"),    # active | ir | suspended | dropped
    "acquired_method": pd.Series(dtype="str"),    # draft | fa | trade | waiver
    "season":          pd.Series(dtype="Int64"),
})

out_path = DATA / "fact_fantasy_teams.parquet"
fact_fantasy_teams.to_parquet(out_path, index=False)
print(f"fact_fantasy_teams: empty schema seed -> {out_path}")
print()
print(fact_fantasy_teams.dtypes.to_string())

## 2 -- Validation

In [ ]:
df = pd.read_parquet(DATA / "fact_fantasy_teams.parquet")
print(f"fact_fantasy_teams: {len(df)} rows (expected 0), {len(df.columns)} columns")
print()
print(df.dtypes.to_string())